In [32]:
import pandas as pd

# Load datasets
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

print("\nTrain columns:")
print(train.columns.tolist())

print("\nTest columns:")
print(test.columns.tolist())

Train shape: (6818, 13)
Test shape: (1705, 12)

Train columns:
['id', 'product_code', 'product_weight_kg', 'fat_content', 'shelf_visibility', 'product_category', 'product_price', 'store_code', 'store_age_years', 'store_size', 'store_location_tier', 'store_format', 'total_sales']

Test columns:
['id', 'product_code', 'product_weight_kg', 'fat_content', 'shelf_visibility', 'product_category', 'product_price', 'store_code', 'store_age_years', 'store_size', 'store_location_tier', 'store_format']


In [33]:
# Categorical columns
categorical_cols = [
    "fat_content",
    "product_category",
    "store_size",
    "store_location_tier",
    "store_format"
]

# Normalize categorical values
for col in categorical_cols:
    train[col] = train[col].str.strip().str.title()
    test[col] = test[col].str.strip().str.title()

print("Categorical normalization complete.")

for col in categorical_cols:
    print(f"{col}: {train[col].nunique()} unique values")

Categorical normalization complete.
fat_content: 2 unique values
product_category: 16 unique values
store_size: 3 unique values
store_location_tier: 3 unique values
store_format: 4 unique values


In [34]:
# Create modeling copies
train_model = train.copy()
test_model = test.copy()

# STORE-JOR has a deterministic store size based on its store format
train_model.loc[
    (train_model["store_code"] == "STORE-JOR") &
    (train_model["store_size"].isna()),
    "store_size"
] = "Small"

test_model.loc[
    (test_model["store_code"] == "STORE-JOR") &
    (test_model["store_size"].isna()),
    "store_size"
] = "Small"

# Preserve genuinely unknown store sizes as an explicit category
train_model["store_size"] = train_model["store_size"].fillna("Missing")
test_model["store_size"] = test_model["store_size"].fillna("Missing")

# Preserve whether product weight was originally missing
train_model["weight_missing"] = (
    train_model["product_weight_kg"].isna().astype(int)
)

test_model["weight_missing"] = (
    test_model["product_weight_kg"].isna().astype(int)
)

print("Modeling data prepared.")

print("\nTrain store_size:")
print(train_model["store_size"].value_counts(dropna=False))

print("\nTest store_size:")
print(test_model["store_size"].value_counts(dropna=False))

print("\nTrain weight_missing:")
print(train_model["weight_missing"].value_counts())

print("\nTest weight_missing:")
print(test_model["weight_missing"].value_counts())

Modeling data prepared.

Train store_size:
store_size
Small      2372
Medium     2218
Missing    1480
Large       748
Name: count, dtype: int64

Test store_size:
store_size
Medium     575
Small      571
Missing    375
Large      184
Name: count, dtype: int64

Train weight_missing:
weight_missing
0    5593
1    1225
Name: count, dtype: int64

Test weight_missing:
weight_missing
0    1399
1     306
Name: count, dtype: int64


In [35]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
TARGET = "total_sales"
ID_COL = "id"

# Separate features and target
X = train_model.drop(columns=[TARGET, ID_COL])
y = train_model[TARGET]

# 80/20 train-validation split
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE
)

print("X_train shape:", X_train.shape)
print("X_valid shape:", X_valid.shape)
print("y_train shape:", y_train.shape)
print("y_valid shape:", y_valid.shape)

X_train shape: (5454, 12)
X_valid shape: (1364, 12)
y_train shape: (5454,)
y_valid shape: (1364,)


In [36]:
# Learn product-level weight medians from the training split
product_weight_median = (
    X_train.groupby("product_code")["product_weight_kg"]
    .median()
)

# Global fallback median from the training split
global_weight_median = X_train["product_weight_kg"].median()

# Impute training weights
X_train["product_weight_kg"] = (
    X_train["product_weight_kg"]
    .fillna(
        X_train["product_code"].map(product_weight_median)
    )
    .fillna(global_weight_median)
)

# Apply the same learned mapping to validation weights
X_valid["product_weight_kg"] = (
    X_valid["product_weight_kg"]
    .fillna(
        X_valid["product_code"].map(product_weight_median)
    )
    .fillna(global_weight_median)
)

print(
    "Remaining missing weights in X_train:",
    X_train["product_weight_kg"].isna().sum()
)

print(
    "Remaining missing weights in X_valid:",
    X_valid["product_weight_kg"].isna().sum()
)

print(
    "\nTraining global weight median:",
    global_weight_median
)

Remaining missing weights in X_train: 0
Remaining missing weights in X_valid: 0

Training global weight median: 12.614


In [37]:
categorical_features = [
    "product_code",
    "fat_content",
    "product_category",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format"
]

print("Categorical features:")
print(categorical_features)

Categorical features:
['product_code', 'fat_content', 'product_category', 'store_code', 'store_size', 'store_location_tier', 'store_format']


In [38]:
from catboost import CatBoostRegressor
from sklearn.metrics import root_mean_squared_error

# Train CatBoost baseline with original features
catboost_model = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    iterations=3000,
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=5,
    random_strength=1.0,
    random_seed=RANDOM_STATE,
    verbose=False,
    allow_writing_files=False
)

catboost_model.fit(
    X_train,
    y_train,
    cat_features=categorical_features,
    eval_set=(X_valid, y_valid),
    early_stopping_rounds=200,
    verbose=False
)

catboost_pred = catboost_model.predict(X_valid)

catboost_rmse = root_mean_squared_error(
    y_valid,
    catboost_pred
)

print(f"CatBoost RMSE: {catboost_rmse:.4f}")
print(
    f"Best iteration: "
    f"{catboost_model.get_best_iteration()}"
)

CatBoost RMSE: 1066.9811
Best iteration: 254


In [39]:
# Proven Optuna CatBoost configuration
best_params = {
    "learning_rate": 0.033649230675604755,
    "depth": 4,
    "l2_leaf_reg": 2.6181170640803217,
    "random_strength": 1.0040061876906814,
    "bagging_temperature": 4.717727204589423,
    "border_count": 180,
    "rsm": 0.9563235629386149
}

optimized_catboost = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    iterations=5000,
    random_seed=RANDOM_STATE,
    verbose=False,
    allow_writing_files=False,
    **best_params
)

optimized_catboost.fit(
    X_train,
    y_train,
    cat_features=categorical_features,
    eval_set=(X_valid, y_valid),
    early_stopping_rounds=200,
    verbose=False
)

optimized_pred = optimized_catboost.predict(X_valid)

optimized_rmse = root_mean_squared_error(
    y_valid,
    optimized_pred
)

print(f"Optimized CatBoost RMSE: {optimized_rmse:.4f}")
print(
    f"Best iteration: "
    f"{optimized_catboost.get_best_iteration()}"
)

Optimized CatBoost RMSE: 1060.6681
Best iteration: 366


In [40]:
# Prepare the full training data
X_full = train_model.drop(columns=[TARGET, ID_COL]).copy()
y_full = train_model[TARGET].copy()

# Learn product-level weight medians from all training data
product_weight_median_full = (
    X_full.groupby("product_code")["product_weight_kg"].median()
)

global_weight_median_full = X_full["product_weight_kg"].median()

# Impute training weights
X_full["product_weight_kg"] = (
    X_full["product_weight_kg"]
    .fillna(X_full["product_code"].map(product_weight_median_full))
    .fillna(global_weight_median_full)
)

# Prepare the test data
X_test = test_model.drop(columns=[ID_COL]).copy()

# Apply the training-derived imputation mapping
X_test["product_weight_kg"] = (
    X_test["product_weight_kg"]
    .fillna(X_test["product_code"].map(product_weight_median_full))
    .fillna(global_weight_median_full)
)

print("Full training shape:", X_full.shape)
print("Test shape:", X_test.shape)
print("Remaining missing values in X_full:", X_full.isna().sum().sum())
print("Remaining missing values in X_test:", X_test.isna().sum().sum())

Full training shape: (6818, 12)
Test shape: (1705, 12)
Remaining missing values in X_full: 0
Remaining missing values in X_test: 0


In [42]:
final_catboost = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    iterations=367,
    learning_rate=0.033649230675604755,
    depth=4,
    l2_leaf_reg=2.6181170640803217,
    random_strength=1.0040061876906814,
    bagging_temperature=4.717727204589423,
    border_count=180,
    rsm=0.9563235629386149,
    random_seed=RANDOM_STATE,
    verbose=False,
    allow_writing_files=False
)

final_catboost.fit(
    X_full,
    y_full,
    cat_features=categorical_features,
    verbose=False
)

print("Final CatBoost model trained successfully.")
print("Iterations:", final_catboost.tree_count_)

Final CatBoost model trained successfully.
Iterations: 367


In [43]:
# Match the test feature order used by the final model
X_test = X_test[final_catboost.feature_names_]

print("Feature alignment:", list(X_test.columns) == list(final_catboost.feature_names_))
print("Test shape:", X_test.shape)
print("Missing values:", X_test.isna().sum().sum())
print("Features:", list(X_test.columns))

Feature alignment: True
Test shape: (1705, 12)
Missing values: 0
Features: ['product_code', 'product_weight_kg', 'fat_content', 'shelf_visibility', 'product_category', 'product_price', 'store_code', 'store_age_years', 'store_size', 'store_location_tier', 'store_format', 'weight_missing']


In [44]:
# Generate test predictions
test_predictions = final_catboost.predict(X_test)

# Create the Kaggle submission
submission = pd.DataFrame({
    "id": test["id"],
    "total_sales": test_predictions
})

# Save submission file
submission.to_csv(
    "submission_catboost_original_optuna.csv",
    index=False
)

print("Submission created successfully.")
print("Shape:", submission.shape)
print("\nFirst 5 predictions:")
print(submission.head())
print("\nPrediction summary:")
print(submission["total_sales"].describe())

Submission created successfully.
Shape: (1705, 2)

First 5 predictions:
          id  total_sales
0  row_00009  2770.666325
1  row_00015  4437.716913
2  row_00019  3104.865088
3  row_00020  3296.501263
4  row_00023  2446.271672

Prediction summary:
count    1705.000000
mean     2206.792321
std      1317.074247
min       -11.556767
25%      1218.942778
50%      2073.001810
75%      3141.969096
max      6795.045333
Name: total_sales, dtype: float64
